In [ ]:
import sys, numpy as np
sys.path.extend(['..'])
import MeshFEM, meshing, mesh_operations, mesh, tri_mesh_viewer

In [ ]:
z_offsets = [[0, 0, 0.5 * z] for z in [-1, 0, 1]]
z_offsets = [[0, 0, 0.5 * z] for z in [    0, 1]]
# z_offsets = [[0, 0, 0.5 * z] for z in [    0,  ]]
z_offsets_before = False

In [ ]:
m = meshing.perforatedSheet([([0.6, 0.6], 0.2), ([0.25, 0.75], 0.1), ([0.75, 0.25], 0.1),
                             ([0.3, 0.3], 0.15)], maxArea=0.01, holeEdgeLen=0.025)
X, F = mesh_operations.reflectMesh(m.vertices(), m.triangles())
numPeriodCellVertices = X.shape[0]
X *= 0.5
X[:, 0:2] += 0.5
if z_offsets_before:
    X, F = mesh_operations.concatenateMeshes((X + t, F) for t in z_offsets)

In [ ]:
def planarDeformedPeriodCellForS(S):
    """
    A linear + periodic deformation field inducing planar macrosopic strain `S - I`
    """
    if ((S == np.identity(2)).all()): return X
    x = np.empty_like(X)
    x[:, 0:2] = X[:, 0:2] @ S.T
    # Add a cartoon periodic fluctuation displacement.
    # Note: this displacement is also stretched by `S`; this is needed
    # ensure identical cartoon deformations from off-midsurface bending
    # and the equivalent pre-stretching.
    x[:, 0:2] += 0.025 * np.column_stack([np.sin(2 * np.pi * X[:, 1]),np.sin(4 * np.pi * X[:, 0])]) @ S.T
    x[:, 2]    = X[:, 2] + 0.05 * np.sin(2 * np.pi * X[:, 0]) * np.sin(2 * np.pi * X[:, 1])
    return x

def planarDeformedTilingForS(S):
    x_cell = planarDeformedPeriodCellForS(S)
    S3d = np.identity(3)
    S3d[0:2, 0:2] = S
    return mesh.Mesh(*mesh_operations.concatenateMeshes([(x_cell + S3d @ t, F) for t in T3d]))

In [ ]:
# Tiling translations: generate a 3x3 planar tiling and duplicate it into multiple sheets
# at various z-offsets.
T = np.array([np.array([s1, s2, 0]) for s1 in [-1, 0, 1] for s2 in [-1, 0, 1]])
T3d = [t for t2d in T for t in [t2d + zo for zo in (z_offsets if not z_offsets_before else [[0, 0, 0]])]]

In [ ]:
def gridCellForVtx(vi):
    nv_column = len(z_offsets) * numPeriodCellVertices
    z_idx = (vi % nv_column) // numPeriodCellVertices
    col_idx = vi // nv_column
    return [col_idx % 3 - 1, col_idx // 3 - 1, z_idx - 1]

In [ ]:
# (theta - sin(theta)) / theta^3
theta_sq_crossover_threshold = 2e-6
def theta_minus_sin_div_theta_cubed(theta):
    theta_sq = theta * theta
    if theta_sq < theta_sq_crossover_threshold: return 1.0 / 6.0 - theta_sq / 120.0
    return (theta - np.sin(theta)) / (theta * theta_sq)

# (1 - cos(theta)) / theta^2
def one_minus_cos_div_theta_sq(theta):
    theta_sq = theta * theta
    if (theta_sq < theta_sq_crossover_threshold): return 0.5 - theta_sq / 24.0
    return (1 - np.cos(theta)) / theta_sq

def sk_inv(w):
    return np.array([[0, -w[2], w[1]], [w[2], 0, -w[0]], [-w[1], w[0], 0]])
def bend_offset(v, grad_w):
    w = grad_w @ v
    theta = np.linalg.norm(w)
    return np.cross(w, v) * one_minus_cos_div_theta_sq(theta) + np.cross(w, np.cross(w, v)) * theta_minus_sin_div_theta_cubed(theta)
import scipy, scipy.linalg
def exp_k_cross(k):
    K = sk_inv(k)
    # theta = np.linalg.norm(k)
    # Note: `np.sinc` is "normalized" and multiplies its argument by `pi`...
    # return np.identity(3) + np.sinc(theta / np.pi) * K + one_minus_cos_div_theta_sq(theta) * (K @ K)
    return scipy.linalg.expm(K)
def rotation_at_offset(v, grad_w):
    w = grad_w @ v
    return exp_k_cross(w)
def bentPosition(x_i, c, grad_w):
    """
    Calculate the bent position of deformed point x_i = S X_i + w_i
    """
    x_mid = x_i.copy()
    x_mid[2] = 0
    x_mid_arc = x_mid + bend_offset(x_mid - c, grad_w)
    return x_mid_arc + x_i[2] * rotation_at_offset(x_mid - c, grad_w) @ [0, 0, 1]

In [ ]:
def bentDeformedPeriodCellForS(S, c, grad_w):
    x_cell = planarDeformedPeriodCellForS(S)
    x_arc = np.empty_like(x_cell)
    for i, x_i in enumerate(x_cell):
        x_arc[i] = bentPosition(x_i, c, grad_w)
    return x_arc

def bentDeformedTranslatedPeriodCell(S, c, t, grad_w):
    # For vertically offset surfaces, the macro state we are passed
    # is `(S_gamma, kappa_gamma, gamma)`.
    S_gamma = S
    gamma = t[2]
    kappa_gamma = np.linalg.norm(grad_w)
    
    # Convert to the equivalent midsurface macrostate (S_0, kappa_0, 0).
    S_0 = (np.identity(2) + (gamma * kappa_gamma) * (grad_w.T @ grad_w / (kappa_gamma**2))[0:2, 0:2]) @ S_gamma
    grad_w_0 = grad_w / (gamma * kappa_gamma + 1)
    
    # Get the deformed base period cell geometry
    c3d = np.pad(S_0 @ c, [(0, 1)])
    x_arc = bentDeformedPeriodCellForS(S_0, c3d, grad_w_0)
    
    # Transform the deformed base cell to the neighboring grid cell
    # (at offset t[0,1] in the rest plane.)
    v = np.pad(S_0 @ t[0:2], [(0, 1)])
    c_trans = bentPosition(c3d + v, c3d, grad_w_0)
    R_v = rotation_at_offset(v, grad_w_0)
    x_arc = (x_arc - c3d) @ R_v.T + c_trans
    
    # Translate the tiling so that the center point `c3d` is at height
    # gamma along the normal [0, 0, 1] at `c`
    return x_arc + [0, 0, gamma]

def bentDeformedTilingForS(S, c, grad_w):
    return mesh.Mesh(*mesh_operations.concatenateMeshes([(bentDeformedTranslatedPeriodCell(S, c, t, grad_w), F) for t in T3d]))

In [ ]:
# Stretching parameters
R = lambda t: np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
S = R(np.pi / 8) @ np.diag([0.9, 1.1]) @ R(-np.pi / 8)

# Bending parameters
unit_vector = lambda t: np.array([np.cos(t), np.sin(t), 0])
k      = -1
a      = unit_vector(0 * np.pi / 2 + np.pi / 4)
a_perp = unit_vector(1 * np.pi / 2 + np.pi / 4)

In [ ]:
mm = planarDeformedTilingForS(np.identity(2))
mmDefo = planarDeformedTilingForS(S)
mmArc = bentDeformedTilingForS(S, [0.0, 0], k * np.outer(a, a_perp)) # - k * np.outer(a_perp, a))

In [ ]:
# Define vertex colors for the mesh to highlight the period cell and fade out the opacity at the periphery
mainColor = [100 / 255, 100 / 255, 200 / 255, 1.0]
peripheralColor = [190 / 255, 200 / 255, 255 / 255, 1.0]
C = np.array([mainColor if gridCellForVtx(v)[0:2] == [0, 0] else peripheralColor for v in range(mm.numVertices())])
def smoothstep(x): return np.select([x < 0, x > 1], [0, 1], default=x*x*(3.0-2.0*x))
dist = np.linalg.norm(mm.vertices()[:, 0:2] - [0.5, 0.5], axis=1)
C[:, 3] = 1 - smoothstep(2 * (dist - 0.75 * np.sqrt(2)))

In [ ]:
import tri_mesh_viewer, mesh
v = tri_mesh_viewer.TriMeshViewer(mm, wireframe=False, scalarField=C)
v.show()

In [ ]:
v.setCameraParams(((0.044388148714364285, -3.573084962775342, 3.1002786330511),
 (0.0, 0.7055853376320855, 0.7052376792583498),
 (0.044388148714364285, -0.25743813495017526, -0.25731128916900514)))

In [ ]:
# Define vertex colors for the mesh to highlight the period cell and fade out the opacity at the periphery
mainColor = [100 / 255, 100 / 255, 200 / 255, 1.0]
peripheralColor = [190 / 255, 200 / 255, 255 / 255, 1.0]
C = np.array([mainColor if gridCellForVtx(v)[0:2] == [0, 0] else peripheralColor for v in range(mm.numVertices())])
def smoothstep(x): return np.select([x < 0, x > 1], [0, 1], default=x*x*(3.0-2.0*x))
dist = np.linalg.norm(mm.vertices()[:, 0:2] - [0.5, 0.5], axis=1)
C[:, 3] = 1 - smoothstep(2 * (dist - 0.75 * np.sqrt(2)))

In [ ]:
for name, m in zip(['rest', 'planar_strain', 'bending_strain'], [mm, mmDefo, mmArc]):
    v.update(mesh=m, scalarField=C)
    orender = v.offscreenRenderer(width=1024,height=1024)
    orender.meshes[0].setColor(C[mm.elements().ravel()])
    orender.render()
    orender.save(f'tiling_{name}.png')

In [ ]:
import tri_mesh_viewer, mesh
v_bend = tri_mesh_viewer.TriMeshViewer(mmArc, scalarField=C)
v_bend.show()

In [ ]:
orender = v_bend.offscreenRenderer(width=1024, height=1024)
orender.meshes[0].setColor(C[mm.elements().ravel()])
orender.setWireframe(1.0, np.pad(C[mm.elements().ravel(), 3:], [(0, 0), (3, 0)]))
orender.render()
orender.image()